# 01 · Neural Network (MLP on handcrafted features)

The **classical** side of the comparison (rubric slides 8–9). No convolution, no pretraining:
hand-designed features (colour + shape + texture), then a multilayer perceptron.
- colour: HSV histogram 8×4×4 = 128-d
- shape: HOG on 128×128 grey = 1,764-d
- texture: uniform LBP histogram = 10-d

Rules: same settings for every `VARIANT`; choose on **val**; evaluate **test** once.
Results go to `results/neural_network/<variant>__<model>__s<seed>.json` and `_preds.csv`.

In [ ]:
VARIANT = "original"     # original, clahe, histeq, gsam, gsam_crop, gsam_light, gsam_light_crop, sam, sam_crop
SEED = 0
FEAT_SIZE = 128

In [ ]:
from pathlib import Path

# ---- paths (preset) -------------------------------------------------------
# project root = the folder that contains pipeline_preprocess/ (works from any sub-folder)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_preprocess").is_dir())
DATA_ROOT = PROJECT_ROOT / "data" / "preprocessing_data"      # original_data/, clahe_data/, gsam_data/, ...
RESULTS_DIR = PROJECT_ROOT / "results" / "neural_network"

# Colab + Google Drive: uncomment and point to the shared folder
# PROJECT_ROOT = Path("/content/drive/MyDrive/dip-project")
# DATA_ROOT = PROJECT_ROOT / "data" / "preprocessing_data"
# RESULTS_DIR = PROJECT_ROOT / "results" / "neural_network"

data_dir = DATA_ROOT / f"{VARIANT}_data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert data_dir.is_dir(), f"{data_dir} not found. Available: {sorted(p.name for p in DATA_ROOT.glob('*_data'))}"
print("data   :", data_dir, sorted(p.name for p in data_dir.iterdir()))
print("results:", RESULTS_DIR)

In [ ]:
import numpy as np
from PIL import Image
from joblib import Parallel, delayed
from skimage.feature import hog, local_binary_pattern

CLASSES = sorted(p.name for p in (data_dir / "train").iterdir())

def extract(path):
    im = Image.open(path).convert("RGB").resize((FEAT_SIZE, FEAT_SIZE))
    hsv, gray = np.asarray(im.convert("HSV")), np.asarray(im.convert("L"))
    color = np.histogramdd(hsv.reshape(-1, 3), bins=(8, 4, 4), range=((0, 256),) * 3)[0].ravel()
    shape = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))
    lbp = np.histogram(local_binary_pattern(gray, 8, 1, "uniform"), bins=10, range=(0, 10), density=True)[0]
    return np.concatenate([color / color.sum(), shape, lbp]).astype(np.float32)

def load(split):
    files = [(f, i) for i, c in enumerate(CLASSES) for f in sorted((data_dir / split / c).glob("*.jpg"))]
    X = np.stack(Parallel(n_jobs=-1)(delayed(extract)(f) for f, _ in files))
    return X, np.array([i for _, i in files]), [f.relative_to(data_dir).as_posix() for f, _ in files]

(Xtr, ytr, _), (Xva, yva, _), (Xte, yte, test_paths) = load("train"), load("val"), load("test")
print("features:", Xtr.shape, Xva.shape, Xte.shape)

## Train: choose the L2 strength on val

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

MODEL_NAME = "mlp"
best = None
for alpha in (1e-4, 1e-3, 1e-2, 1e-1):
    clf = make_pipeline(StandardScaler(),
                        MLPClassifier(hidden_layer_sizes=(512, 128), alpha=alpha, max_iter=300,
                                      early_stopping=True, random_state=SEED)).fit(Xtr, ytr)
    f1 = f1_score(yva, clf.predict(Xva), average="macro")
    print(f"alpha={alpha:g}  val F1 {f1:.3f}")
    if best is None or f1 > best[0]:
        best = (f1, alpha, clf)
val_f1, alpha, clf = best
CONFIG = {"alpha": alpha, "hidden": [512, 128], "features": "hsv128+hog1764+lbp10", "val_f1": val_f1}

## Test (once)

In [ ]:
y_true, y_pred = yte, clf.predict(Xte)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
test_metrics = {"accuracy": accuracy_score(y_true, y_pred), "precision": p, "recall": r, "f1": f1}
print(f"{VARIANT} / {MODEL_NAME}:", {k: round(v, 4) for k, v in test_metrics.items()})
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=CLASSES, xticks_rotation=60,
                                        normalize="true", values_format=".2f", colorbar=False)
plt.title(f"{VARIANT} / {MODEL_NAME}"); plt.tight_layout(); plt.show()

In [ ]:
import json, pandas as pd
run = f"{VARIANT}__{MODEL_NAME}__s{SEED}"
pd.DataFrame({"path": test_paths, "y_true": [CLASSES[i] for i in y_true], "y_pred": [CLASSES[i] for i in y_pred]}
            ).to_csv(RESULTS_DIR / f"{run}_preds.csv", index=False)
json.dump({"variant": VARIANT, "model": MODEL_NAME, "seed": SEED, **test_metrics, **CONFIG},
          open(RESULTS_DIR / f"{run}.json", "w"), indent=2)
print("saved:", RESULTS_DIR / run)